# Training and inference energy measurements

This notebook runs ten repetitions with an idle baseline only when a hardware energy counter is available. Linux RAPL and a local Scaphandre Prometheus endpoint are supported as actual energy sources. CodeCarbon is exposed as an explicitly labelled estimate unless an actual sensor is independently verified. Windows generally does not expose Linux RAPL; without a sensor, the notebook records an unavailable status and does not invent energy values.


In [ ]:
import json
import re
import sys
import time
from pathlib import Path
from urllib.request import urlopen

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 and Notebook 04 first.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_DIR = ROOT / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPETITIONS = 10
RUN_CODECARBON_ESTIMATES = False
SCAPHANDRE_METRICS_URL = "http://localhost:8080/metrics"

df_ml_ready = pd.read_parquet(PROCESSED_DIR / "prepared_training_dataset.parquet")
manifest = json.loads((PROCESSED_DIR / "feature_manifest.json").read_text(encoding="utf-8"))
model_metadata = json.loads((PROCESSED_DIR / "models" / "training_metadata.json").read_text(encoding="utf-8"))
feature_columns = model_metadata["feature_columns"]
model_names = ["logistic_regression", "random_forest", "xgboost"]
models = {
    name: joblib.load(PROCESSED_DIR / "models" / f"{name}.joblib")
    for name in model_names
}
X = df_ml_ready[feature_columns].astype(float)
y = df_ml_ready["energy_winner"].astype(int)
sample_size = min(256, len(X))
inference_X = X.sample(n=sample_size, random_state=42)

def find_rapl_counters():
    if not sys.platform.startswith("linux"):
        return []
    counters = []
    for energy_file in Path("/sys/class/powercap").glob("intel-rapl:*/energy_uj"):
        if not re.fullmatch(r"intel-rapl:\\d+", energy_file.parent.name):
            continue
        range_file = energy_file.parent / "max_energy_range_uj"
        if range_file.is_file():
            try:
                int(energy_file.read_text().strip())
                counters.append((energy_file, int(range_file.read_text().strip())))
            except (OSError, ValueError):
                continue
    return counters

def read_rapl_joules(counters):
    return sum(int(energy_file.read_text().strip()) for energy_file, _ in counters) / 1_000_000

def rapl_delta_joules(start_joules, counters):
    current = [int(energy_file.read_text().strip()) for energy_file, _ in counters]
    return sum(
        ((value / 1_000_000) - (start_joules[index] / 1_000_000)) % (limit / 1_000_000)
        for index, (value, (_, limit)) in enumerate(zip(current, counters))
    )

def read_scaphandre_joules(url=SCAPHANDRE_METRICS_URL):
    with urlopen(url, timeout=1) as response:
        metrics = response.read().decode("utf-8")
    for line in metrics.splitlines():
        if line.startswith("scaph_host_energy_microjoules_total"):
            value = float(line.rsplit(None, 1)[-1])
            return value / 1_000_000
    raise ValueError("Scaphandre endpoint has no scaph_host_energy_microjoules_total counter.")

rapl_counters = find_rapl_counters()
scaphandre_available = False
try:
    read_scaphandre_joules()
    scaphandre_available = True
except (OSError, ValueError, TimeoutError):
    pass

if rapl_counters:
    energy_backend = "RAPL"

    def read_energy():
        return read_rapl_joules(rapl_counters)

    def delta_energy(start):
        current = [int(energy_file.read_text().strip()) for energy_file, _ in rapl_counters]
        return sum(
            ((value / 1_000_000) - (start[index] / 1_000_000)) % (limit / 1_000_000)
            for index, (value, (_, limit)) in enumerate(zip(current, rapl_counters))
        )

    def take_counter_snapshot():
        return [int(energy_file.read_text().strip()) for energy_file, _ in rapl_counters]
elif scaphandre_available:
    energy_backend = "Scaphandre host counter"
    take_counter_snapshot = read_scaphandre_joules
    delta_energy = lambda start: read_scaphandre_joules() - start
else:
    energy_backend = None

def measure_with_codecarbon_estimate(workload):
    """Optional CodeCarbon estimate; never label this return value as a sensor measurement."""
    from codecarbon import EmissionsTracker

    tracker = EmissionsTracker(
        project_name="energy_aware_recommendation",
        save_to_file=False,
        save_to_logger=False,
        save_to_api=False,
        log_level="error",
    )
    tracker.start()
    try:
        workload()
    finally:
        tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed
    return {
        "energy_joules_estimate": float(energy_kwh) * 3_600_000,
        "actual_measurement": False,
        "method": "CodeCarbon estimate (not a hardware-verified measurement)",
    }

print(f"Measurement backend: {energy_backend or 'none'}")
print(f"CodeCarbon available for explicitly labelled estimates: {__import__('importlib').util.find_spec('codecarbon') is not None}")


In [ ]:
def measure_one_repetition(model_name, task, task_name, repetition):
    counter_start = take_counter_snapshot()
    started = time.perf_counter()
    task()
    elapsed = time.perf_counter() - started
    gross_joules = delta_energy(counter_start)

    baseline_start = take_counter_snapshot()
    time.sleep(elapsed)
    baseline_joules = delta_energy(baseline_start)
    return {
        "model": model_name,
        "task": task_name,
        "repetition": repetition,
        "backend": energy_backend,
        "elapsed_seconds": elapsed,
        "gross_energy_joules": gross_joules,
        "idle_baseline_joules": baseline_joules,
        "net_energy_joules": gross_joules - baseline_joules,
        "n_inference_pairs": sample_size if task_name == "inference" else np.nan,
        "actual_measurement": True,
    }

measurement_rows = []
if energy_backend is None:
    status = {
        "actual_measurements_available": False,
        "status": "unavailable",
        "reason": "No RAPL counter or local Scaphandre host energy counter was detected.",
        "platform": sys.platform,
        "requested_repetitions": REPETITIONS,
        "completed_repetitions_per_model_task": 0,
        "completed_measurements": 0,
        "codecarbon_note": "CodeCarbon estimates are not written as actual energy measurements.",
    }
    print(status["reason"])
else:
    for model_name, model in models.items():
        def train_once(model=model):
            estimator = clone(model)
            estimator.fit(X, y)

        def infer_once(model=model):
            model.predict(inference_X)

        for task_name, task in (("training", train_once), ("inference", infer_once)):
            for repetition in range(1, REPETITIONS + 1):
                measurement_rows.append(measure_one_repetition(model_name, task, task_name, repetition))
    status = {
        "actual_measurements_available": True,
        "status": "measured",
        "backend": energy_backend,
        "requested_repetitions": REPETITIONS,
        "completed_repetitions_per_model_task": REPETITIONS,
        "completed_measurements": len(measurement_rows),
        "platform": sys.platform,
        "baseline": "Same-duration idle interval subtracted per repetition; raw and net values retained.",
    }

estimate_rows = []
if RUN_CODECARBON_ESTIMATES:
    for model_name, model in models.items():
        for task_name, task in (
            ("training", lambda model=model: clone(model).fit(X, y)),
            ("inference", lambda model=model: model.predict(inference_X)),
        ):
            for repetition in range(1, REPETITIONS + 1):
                estimate = measure_with_codecarbon_estimate(task)
                estimate_rows.append({
                    "model": model_name,
                    "task": task_name,
                    "repetition": repetition,
                    **estimate,
                })
pd.DataFrame(estimate_rows).to_csv(RESULTS_DIR / "energy_estimates_codecarbon.csv", index=False)

measurement_columns = [
    "model", "task", "repetition", "backend", "elapsed_seconds", "gross_energy_joules",
    "idle_baseline_joules", "net_energy_joules", "n_inference_pairs", "actual_measurement",
]
measurements = pd.DataFrame(measurement_rows, columns=measurement_columns)
measurements.to_csv(RESULTS_DIR / "energy_measurements.csv", index=False)
summary = (
    measurements.groupby(["model", "task", "backend"], as_index=False)
    .agg(
        repetitions=("net_energy_joules", "count"),
        mean_net_energy_joules=("net_energy_joules", "mean"),
        std_net_energy_joules=("net_energy_joules", "std"),
        mean_gross_energy_joules=("gross_energy_joules", "mean"),
        mean_idle_baseline_joules=("idle_baseline_joules", "mean"),
        inference_pairs=("n_inference_pairs", "mean"),
    )
)
summary.to_csv(RESULTS_DIR / "energy_measurement_summary.csv", index=False)
(RESULTS_DIR / "energy_measurement_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
if not summary.empty:
    display(summary)
print(json.dumps(status, indent=2))
